<a href="https://colab.research.google.com/github/emmannuelltr/Analitica-de-Negocios/blob/main/carpeta2/NaibeBayes1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**CASO DE ESTUDIO**
En este documento se desarrolla y analiza un modelo Naive Bayes, para la deteccion de patrones que permitan clasificar los clientes con alto riesgo de incurrir en lavado de activos utilizando variables financieras y socioeconómicas para para apoyar la identificación temprana de clientes de riesgo. De acuerdo con lo anterior las variables financieras y socio económicas son:
**Edad**: es la edad que posee el usuario a evaluar como parte de los controles legales
**ingresos_usd**: Ingresos recibidos por una persona como resultado de una actividad comercial
**gastos_usd**: Son los gastos mensuales de la persona
**num_tarjetas_credito** Indica nivel de acceso financiero y posible perfil de riesgo
**monto_transado_tarjetas_usd** Detecta patrones de consumo o movimientos inusuales
**porcentaje_crecimiento_patrimonio** Útil para identificar crecimientos anómalos

In [ ]:
import numpy as np #Libria numérica por excelencia
import pandas as pd

#Librerias especifificas
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix

1. Se carga los datos de trabajo de la base de datos

In [ ]:
nxl="/content/sample_data/2. LavadoActivos.xlsx"
XDB=pd.read_excel(nxl,sheet_name=0)
XDB.head(100)

#Seleccionamos variables de trabajo
XD= XDB[['edad', 'ingresos_usd', 'gastos_usd', 'num_tarjetas_credito',
        'monto_transado_tarjetas_usd', 'porcentaje_crecimiento_patrimonio']]
XD.head()
yd=XDB[['lavado_activos']]
yd.head()

,lavado_activos
0,1
1,1
2,0
3,0
4,0


2. Implementamos Modelo Naive Bayes

In [ ]:
mnb=GaussianNB()
mnb.fit(XD,yd) #Ajustar variables entradas - salidas

#Mostramos medias
u=mnb.theta_
sigma=mnb.var_;sigma=np.sqrt(sigma)
np.set_printoptions(suppress=True, precision=2) #Para que me de los numeros normales y no cientficos
print("Las medias son:")
print('edad', 'ingresos_usd', 'gastos_usd', 'num_tarjetas_credito',
        'monto_transado_tarjetas_usd', 'porcentaje_crecimiento_patrimonio')
print(u)
print("las desviaciones son:")
print(sigma)

Las medias son:
edad ingresos_usd gastos_usd num_tarjetas_credito monto_transado_tarjetas_usd porcentaje_crecimiento_patrimonio
[[   45.47  6690.73  3601.75     1.95  3090.46    44.69]
 [   44.3  70262.3  38707.28     2.02 33835.62    44.82]]
las desviaciones son:
[[    14.71   6206.96   3219.02      2.54   2456.86     14.17]
 [    15.05 113412.26  65902.56      2.56  62991.39     15.48]]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


**Análisis de resultados**
Mirando los resultados del modelo, queda claro que lo que separa a un cliente normal de uno sospechoso no es la edad ni cuántas tarjetas tiene, sino el volumen de dinero que mueve. Mientras que un cliente promedio tiene ingresos de unos 6690 USD, el grupo de riesgo dispara esa cifra de en promedio 70262 USD, lo cual es una gran diferencia . Lo mismo pasa con el monto transado en tarjetas, es casi diez veces mayor en los casos de riesgo. Lo curioso es que la edad se mantiene igual en ambos grupos en promedio 45 años, así que el modelo nos dice que el peligro no está en la edad, sino en esa transaccionalidad desproporcionada.

3. Se procede con la evaluacion del modelo, para la evaluacion de este tipo de modelo se utiliza la matriz de confusión

In [ ]:
ydp=mnb.predict(XD) #Esto es lo que el modelo aprende - ydp de pronóstico
cm=confusion_matrix(yd,ydp)
print(cm)

#Se determinan las metricas de la matriz de confusión
VN=cm[0,0];FP=cm[1,0];FN=cm[1,0]; VP=cm[1,1];TDatos=len(XDB)

#1. Exactitud: funcionamiento general del modelo-
Ex=(VP+VN)/TDatos
print("Exactitud: ",Ex)

#2. Tasa Error: %Fallos del Modelo
TEr=(FP+FN)/TDatos
print("Tasa Error: ",TEr)

#3. Sensibilidad: como se comporto con respecto a los positivos
Se= VP/(VP+FN)
print("Sensibilidad: ",Se)

#4. Especificidad: como se comporta diagnosticando negativos
Es= VN/(VN+FP)
print("Especificidad: ",Es)

#5. Precisión: Es una version de como se comporta el modelo frente a los positivos solamente
Pr=VP/(VP+FP)
print("Precisión: ",Pr)


[[2179   44]
 [ 145  782]]
Exactitud:  0.94
Tasa Error:  0.09206349206349207
Sensibilidad:  0.8435814455231931
Especificidad:  0.9376075731497419
Precisión:  0.8435814455231931


**Análisis de datos**
De acuerdo con los resultados, podemos observar de manera general que el modelo alcanzó una exactitud del 94%, lo que indica un desempeño muy sólido al clasificar a los clientes entre sospechosos y normales. Se destaca la precisión del 84.4%, lo que nos dice que el modelo funciona bastante bien a la hora de identificar correctamente los casos reales de riesgo de lavado. Igualmente, se nota un gran comportamiento para detectar a los clientes que no representan una amenaza, tal como lo demuestra la especificidad del 93.8%. Es importante mencionar que, aunque el modelo es muy robusto, todavía existe un pequeño margen de mejora reflejado en la tasa de error del 9.2%, lo cual se evidencia en la sensibilidad (84.4%), sugiriendo que aún hay algunos casos sospechosos que el modelo podría estar omitiendo y que requieren atención para optimizar la detección temprana

4. Evaluamos un solicitante

In [ ]:
XP=(50,3000,3000,2,3000,1000)
ydc=mnb.predict([XP])
print(ydc)

if  ydc==1:
  print("el solicitante es sospechoso de lavado de activos")
else:
  print("el solicitante es no es sospechoso de lavado de activos")

[1]
el solicitante es sospechoso de lavado de activos


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but GaussianNB was fitted with feature names
  warnings.warn(
